In [ ]:
import os
import Sleep_Scripts.ASCII_to_binary_2ch_downsampled as ASCII
import Sleep_Scripts.Practical_scripts as misc
import mne


In [ ]:
# Point to the folder containing the original ASCII brainvision .dat and .vhdr files
base_dir = "D:/Intracranial_sleep_data"
# Point to the output folder where the downsampled data is stored
binary_path = "D:/EEG_Data_stage/"
# Name of the converted file folder
file_type = "converted_intra_upscale_strips"
subjects = os.listdir(base_dir)

In [ ]:
# Define subjects (keys) and the electrodes you'd like to use (values)
# To make things easier downstream, make the hippocampus contact index 0 and the cortex contact index 1
electrodes = {"2":["TL05", "TLR06"],
              "7":["TL03", "TLR06"],
              "15":["TL05", "TLR01"], 
              "28":["TL07", "TLR06"], 
              "31":["TL05", "TLR12"], 
              "87":["TL05", "TLR01"]}

In [ ]:
# Run through each subject folder
for subject in subjects:
    if subject in electrodes.keys():
        # Run through each file
        files = os.listdir(os.path.join(base_dir, subject, 'iEEG'))
        # Each vhdr file gets stored in a list
        vhdr_files = [a for a in files if 'vhdr' in a]
        # The ASCII gets converted to binary, here the output file is prepared
        binary_file = os.path.join(binary_path, subject, 'iEEG', file_type)
        # Selecting vhdr files automatically converts the accompanying .dat files
        for vhdr_file in vhdr_files:
            print(f'Converting {subject}-{vhdr_file}')
            # Convert each file, selecting which channels you'd like to keep, and downsampling from 1000 to 250 hz (divided by 4)
            ASCII.convert_brainvision_ascii(os.path.join(base_dir, subject, 'iEEG', vhdr_file),
                                            binary_file, electrodes[subject], downsample_factor=4)

In [ ]:
converted_subjects = os.listdir("C:/EEG_Data_stage/7/iEEG/converted")

In [ ]:
# for subject in converted_subjects:
#     print(f"Linking nights in subject {subject}")
#     # Link all night fragments together to form a full night
misc.link_sections(os.path.join(converted_subjects), "brainvision")

In [ ]:
combined_night = "D:/converted_sleep_data/7/combined_nights/night1.vhdr"
raw = mne.io.read_raw_brainvision(combined_night)

In [ ]:
# Convert channels to their correct types e.g. EOG is marked as eog instead of eeg
raw = misc.convert_channel_types(raw)
print(raw.get_channel_types())

In [ ]:
# Plot the data to visualize the eeg data interactively
fig = raw.plot(duration=120,
    scalings=dict(eeg=1e-4) 
)
fig.subplots_adjust(top=0.9)
plt.show(block=True)

In [ ]:
# Convert brainvision files to edf, to be used in u-sleep
print(os.listdir(binary_path))
for folder in os.listdir(binary_path):
    if folder not in ["plots", "bandpower", "line_per_state", "67"] and folder in ["86", "87"]:
        for file in os.listdir(os.path.join(binary_path, folder, "iEEG", 'converted_ec')):
            if 'vhdr' in file:
                print(f"converting {file} to edf")
                misc.convert_binary_brainvision(os.path.join(binary_path, folder, "iEEG", 'converted_ec', file))
# misc.convert_binary_brainvision(os.path.join(base_dir, '7', 'iEEG', 'converted', '2_night2_04.vhdr'))


In [ ]:
"""After annotating the data with the u-sleep web interface, we can start converting and downsampling the files
Keeping a central, an occipital an eog and an emg channel """
for subject in os.listdir(base_dir):
    data = os.path.join(base_dir, subject, 'iEEG')
    out_path = os.path.join(data, 'converted_prep')
    for file in os.listdir(data):
        if "vhdr" in file:
            print(f"converting {file}")
            ASCII.convert_brainvision_ascii(os.path.join(data, file), out_path, ['EOG1', 'Cz', 'Oz', 'EMG1'])
            print(f"converted {file}")
            

In [ ]:
# data = os.path.join(base_dir, '2', 'iEEG')
convert = "C:/EEG_Data_stage/2/iEEG/converted_intra/edf_files"
misc.link_sections(convert, "edf")